In [9]:
import os
hf_api =""
OPENAI_API_KEY_HARDCODED = ""
TAVILY_API_KEY_HARDCODED = ""
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY_HARDCODED
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY_HARDCODED



In [ ]:
!pip install -U kaleido tavily-python smolagents

In [8]:
# %% EARTHQUAKE VISUALIZATION AGENT (COMPLETE CODE)
from tavily import TavilyClient
from smolagents import tool, Tool, CodeAgent, OpenAIServerModel
import os
from dotenv import load_dotenv
from PIL import Image
import pandas as pd
from typing import Any
from smolagents.utils import make_image_url, encode_image_base64
import kaleido

load_dotenv(override=True)

# ======== CORE TOOLS ========
@tool
def web_search(query: str) -> str:
    """Searches the web for earthquake data.

    Args:
        query: Search query about seismic activity
    """
    tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    response = tavily_client.search(query)
    return str(response["results"])

class VisitWebpageTool(Tool):
    name = "visit_webpage"
    description = "Reads webpage content about geological data"
    inputs = {"url": {"type": "string", "description": "URL of seismic data source"}}
    output_type = "string"

    def forward(self, url: str) -> str:
        try:
            import re
            import requests
            from markdownify import markdownify
            from requests.exceptions import RequestException
            from smolagents.utils import truncate_content

            response = requests.get(url, timeout=20)
            response.raise_for_status()
            markdown_content = markdownify(response.text).strip()
            markdown_content = re.sub(r"\n{3,}", "\n\n", markdown_content)
            return truncate_content(markdown_content, 40000)

        except requests.exceptions.Timeout:
            return "Request timed out"
        except RequestException as e:
            return f"HTTP Error: {str(e)}"
        except Exception as e:
            return f"Error: {str(e)}"

# ======== AGENT DEFINITIONS ========
quake_web_agent = CodeAgent(
    model=OpenAIServerModel("gpt-4.1-2025-04-14", max_completion_tokens=8096),
    tools=[web_search, VisitWebpageTool()],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "plotly.express",
        "numpy",
        "datetime",
        "requests",
        "pandas",
        "plotly.express",
        "os"
        "matplotlib.pyplot",
        "io",
        "base64",
        "posixpath"
    ],
    max_steps=10,
    name="quake_web_agent",
    description="Collects real-time seismic data from web sources"
)
quake_web_agent.logger.console.width = 66



quake_manager = CodeAgent(
    model=OpenAIServerModel("gpt-4.1-2025-04-14", max_tokens=8096),
    tools=[],
    managed_agents=[quake_web_agent],
    additional_authorized_imports=[
        "geopandas",
        "plotly",
        "plotly.express",
        "numpy",
        "datetime",
        "requests",
        "pandas",
        "plotly.express",
        "os",
        "matplotlib.pyplot",
        "posixpath"
    ],
    planning_interval=5,
    verbosity_level=2,
    max_steps=15
)
quake_manager.logger.console.width = 66

# ======== EXECUTION ========
if os.path.exists(r"content/seismic_map.png"):
    os.remove(r"content/seismic_map.png")

quake_manager.run(f"""
Analyze global seismic activity from the past 30 days.
Create a world map using px.scatter_geo where:
- Marker size corresponds to earthquake magnitude
- Color represents focal depth
- Hover text shows location and date
- Use real data from USGS/EMSC sources

Save to content/seismic_map.png and return the plot.

Example code structure:
import plotly.express as px
fig = px.scatter_geo(df, lat='latitude', lon='longitude',
                     size='magnitude', color='depth_km',
                     hover_name='location')
fig.write_image(r'content/seismic_map.png')
final_answer(fig)
""")

# Display results
if "fig" in quake_manager.python_executor.state:
    fig = quake_manager.python_executor.state["fig"]
    fig.update_layout(
        title="Global Seismic Activity (Past 30 Days)",
        width=1000,
        height=800,
        geo=dict(
            showland=True,
            landcolor="rgb(243, 243, 243)",
            countrycolor="rgb(204, 204, 204)"
        )
    )
    fig.show()
else:
    print("Execution failed. Final output:", quake_manager.final_answer)

╭─────────────────────────── New run ────────────────────────────╮
│                                                                │
│ Analyze global seismic activity from the past 30 days.         │
│ Create a world map using px.scatter_geo where:                 │
│ - Marker size corresponds to earthquake magnitude              │
│ - Color represents focal depth                                 │
│ - Hover text shows location and date                           │
│ - Use real data from USGS/EMSC sources                         │
│                                                                │
│ Save to content/seismic_map.png and return the plot.           │
│                                                                │
│ Example code structure:                                        │
│ import plotly.express as px                                    │
│ fig = px.scatter_geo(df, lat='latitude', lon='longitude',      │
│                      size='magnitude', color='depth_km',       │
│                      hover_name='location')                    │
│ fig.write_image(r'content/seismic_map.png')                    │
│ final_answer(fig)                                              │
│                                                                │
╰─ OpenAIServerModel - gpt-4.1-2025-04-14 ───────────────────────╯

────────────────────────── Initial plan ──────────────────────────
Here are the facts I know and the plan of action that I will 
follow to solve the task:
```
## 1. Facts survey

### 1.1. Facts given in the task

- The analysis should cover global seismic (earthquake) activity 
for the past 30 days.
- Data source: Use real data from USGS and/or EMSC sources.
- Plot requirements:
    - Use px.scatter_geo (from plotly.express).
    - Marker size should correspond to earthquake magnitude.
    - Marker color should represent focal depth ("depth", likely 
in km).
    - Hover text should show location and date.
- The output plot must be saved to content/seismic_map.png and the
plot should be returned as the answer.
- Example code structure is provided as a guide.

### 1.2. Facts to look up

- Latest global seismic/earthquake dataset from the past 30 days:
    - Required fields: latitude, longitude, magnitude, depth in 
km, date/time, location (place name).
    - Possible sources for data download:
        - USGS Earthquake Catalog: 
https://earthquake.usgs.gov/earthquakes/feed/v1.0/geojson.php
        - EMSC Earthquake Data: 
https://www.emsc-csem.org/Earthquake/
    - Need to determine the exact download link and/or API method 
for retrieving events globally for the last 30 days.
- Data format from chosen source(s) to map their columns/fields to
the required structure:
    - Check field names and how to extract: latitude, longitude, 
magnitude, depth (km), location, and datetime.
- Details on saving figures using plotly.express with .write_image
(does it require additional dependencies, e.g., kaleido or orca?).
- Full list of country or region names (or geolocation place 
field) provided in the earthquake dataset for mapping as hover 
text.
- The datetime format used by source (for formatting hover text).

### 1.3. Facts to derive

- Processed dataframe where:
    - Each row is an earthquake within the past 30 days.
    - Columns: latitude, longitude, magnitude, depth_km, location,
date.
- Mapping/matching of source data field names to those required by
Plotly for the map.
- Formatted hover text per plotly's px.scatter_geo "hover_name" 
parameter such that it displays both location and date.
- Validation: ensure all required fields are present and properly 
formatted.
- Exported image file saved to the correct path: 
content/seismic_map.png.
- The final plotly figure object for returning.

---

## 2. Plan

1. Determine which data source (USGS or EMSC) to use for global 
earthquake data from the last 30 days, and look up the 
query/download/API method.
2. Download and load the earthquake data for the past 30 days, 
ensuring it covers the following columns: latitude, longitude, 
magnitude, depth, location, and datetime (using USGS/EMSC as 
needed).
3. Examine the data structure and map its columns to the required 
structure (latitude, longitude, magnitude, depth_km, location, 
date).
4. Clean and preprocess the dataset:
    a. Filter for events within the last 30 days if the data 
includes more.
    b. Reformat or parse the date/time field so it can be 
displayed as a concise string in hover text.
    c. Ensure latitude/longitude, magnitude, and depth values are 
numeric and usable for plotting.
5. Construct a dataframe matching the field requirements for 
px.scatter_geo (latitude, longitude, magnitude, depth_km, 
location, date).
6. Create a px.scatter_geo plot:
    a. Set marker size to magnitude.
    b. Set color to depth in km.
    c. Set hover text to display location and date.
7. Ensure any required dependencies for saving plotly images are 
installed and configured (e.g., kaleido).
8. Save the plot to content/seismic_map.png using fig.write_image.
9. Return the plotly figure as the final answer.

```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ───────────────────────────────────────
Thought:                                                          
Step 1 is to get a current global earthquake dataset for the past 
30 days. The most straightforward source is the USGS Earthquake   
Catalog, which has a GeoJSON feed for "Past 30 days" worldwide:   
https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_mont
h.geojson                                                         
                                                                  
I'll download this using requests, parse its 'features' JSON list,
and extract for each event:                                       
- coordinates (longitude, latitude, depth)                        
- properties: mag, place, time (epoch ms)                         
                                                                  
I'll load these into a pandas DataFrame and print the column names
and a few rows to inspect the structure.                          
Code:                                                             
```py                                                             
import requests                                                   
import pandas as pd                                               
import datetime                                                   
                                                                  
# Download most recent global earthquake data (past 30 days)      
url =                                                             
"https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_mon
th.geojson"                                                       
resp = requests.get(url)                                          
data = resp.json()                                                
                                                                  
# Extract relevant fields                                         
records = []                                                      
for feature in data["features"]:                                  
    coords = feature["geometry"]["coordinates"]  # [lon, lat,     
depth]                                                            
    props = feature["properties"]                                 
    record = {                                                    
        "longitude": coords[0],                                   
        "latitude": coords[1],                                    
        "depth_km": coords[2],                                    
        "magnitude": props["mag"],                                
        "location": props["place"],                               
        "time": props["time"]  # epoch ms                         
    }                                                             
    [38;2;230;237;243;48;2;13;17;23

─ Executing parsed code: ─────────────────────────────────────── 
  import requests                                                 
  import pandas as pd                                             
  import datetime                                                 
                                                                  
  # Download most recent global earthquake data (past 30 days)    
  url =                                                           
  "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all  
  _month.geojson"                                                 
  resp = requests.get(url)                                        
  data = resp.json()                                              
                                                                  
  # Extract relevant fields                                       
  records = []                                                    
  for feature in data["features"]:                                
      coords = feature["geometry"]["coordinates"]  # [lon, lat,   
  depth]                                                          
      props = feature["properties"]                               
      record = {                                                  
          "longitude": coords[0],                                 
          "latitude": coords[1],                                  
          "depth_km": coords[2],                                  
          "magnitude": props["mag"],                              
          "location": props["place"],                             
          "time": props["time"]  # epoch ms                       
      }                                                           
      records.append(record)                                      
                                                                  
  df = pd.DataFrame(records)                                      
  print(df.head())                                                
  print(df.dtypes)                                                
 ────────────────────────────────────────────────────────────────

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ───────────────────────────────────────
Thought:                                                          
The error occurs because some earthquake records have negative    
magnitudes, which are invalid for marker size in px.scatter_geo   
(it must be >=0).                                                 
To fix this, I will filter the dataframe to include only rows     
where "magnitude" is greater than zero before plotting.           
                                                                  
Code:                                                             
```py                                                             
# Filter data: only keep earthquakes with positive magnitude      
plot_df = df[df["magnitude"] > 0].copy()                          
                                                                  
# Try plotting again                                              
fig = px.scatter_geo(plot_df,                                     
                     lat="latitude",                              
                     lon="longitude",                             
                     size="magnitude",                            
                     color="depth_km",                            
                     color_continuous_scale="Viridis",            
                     hover_name="hover",                          
                     size_max=16,                                 
                     projection="natural earth",                  
                     title="Global Seismic Activity (Past 30      
Days)")                                                           
                                                                  
fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})              
                                                                  
# Save as PNG                                                     
fig.write_image("content/seismic_map.png")                        
                                                                  
final_answer(fig)                                                 
```<end_code>                                                     

─ Executing parsed code: ─────────────────────────────────────── 
  # Filter data: only keep earthquakes with positive magnitude    
  plot_df = df[df["magnitude"] > 0].copy()                        
                                                                  
  # Try plotting again                                            
  fig = px.scatter_geo(plot_df,                                   
                       lat="latitude",                            
                       lon="longitude",                           
                       size="magnitude",                          
                       color="depth_km",                          
                       color_continuous_scale="Viridis",          
                       hover_name="hover",                        
                       size_max=16,                               
                       projection="natural earth",                
                       title="Global Seismic Activity (Past 30    
  Days)")                                                         
                                                                  
  fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})            
                                                                  
  # Save as PNG                                                   
  fig.write_image("content/seismic_map.png")                      
                                                                  
  final_answer(fig)                                               
 ────────────────────────────────────────────────────────────────

Out - Final answer: Figure({
    'data': [{'geo': 'geo',
              'hovertemplate': ('<b>%{hovertext}</b><br><br>mag' 
... '%{marker.color}<extra></extra>'),
              'hovertext': array(['2 km NW of The Geysers, CA 
(2025-05-03)',
                                  '54 km WNW of Valdez, Alaska 
(2025-05-03)',
                                  '5 km WSW of Anza, CA 
(2025-05-03)', ...,
                                  '14 km ENE of Yerington, Nevada 
(2025-04-03)',
                                  '42 km W of Chignik Lake, Alaska
(2025-04-03)',
                                  '42 km SE of Unalaska, Alaska 
(2025-04-03)'], dtype=object),
              'lat': array([38.7879982 , 61.3065    , 33.5325    ,
..., 39.0536    , 56.29133333,
                            53.565     ]),
              'legendgroup': '',
              'lon': array([-122.77083588, -147.2872    , 
-116.7186667 , ..., -119.0222    ,
                            -159.43766667, -166.1687    ]),
              'marker': {'color': array([ 0.98000002, 14.5       ,
5.83      , ...,  3.7       ,  3.81      ,
                                         51.        ]),
                         'coloraxis': 'coloraxis',
                         'size': array([1.54, 1.6 , 0.75, ..., 1.1
, 0.07, 1.5 ]),
                         'sizemode': 'area',
                         'sizeref': 0.02890625,
                         'symbol': 'circle'},
              'mode': 'markers',
              'name': '',
              'showlegend': False,
              'type': 'scattergeo'}],
    'layout': {'coloraxis': {'colorbar': {'title': {'text': 
'depth_km'}},
                             'colorscale': [[0.0, '#440154'], 
[0.1111111111111111,
                                            '#482878'], 
[0.2222222222222222,
                                            '#3e4989'], 
[0.3333333333333333,
                                            '#31688e'], 
[0.4444444444444444,
                                            '#26828e'], 
[0.5555555555555556,
                                            '#1f9e89'], 
[0.6666666666666666,
                                            '#35b779'], 
[0.7777777777777778,
                                            '#6ece58'], 
[0.8888888888888888,
                                            '#b5de2b'], [1.0, 
'#fde725']]},
               'geo': {'center': {},
                       'domain': {'x': [0.0, 1.0], 'y': [0.0, 
1.0]},
                       'projection': {'type': 'natural earth'}},
               'legend': {'itemsizing': 'constant', 
'tracegroupgap': 0},
               'margin': {'b': 0, 'l': 0, 'r': 0, 't': 40},
               'template': '...',
               'title': {'text': 'Global Seismic Activity (Past 30
Days)'}}
})

[Step 4: Duration 5.12 seconds| Input tokens: 17,285 | Output 
tokens: 1,080]